# lyrkl pipeline test notebook

Progressive test of every lyrkl module and pipeline stage.
Run cells top-to-bottom. The LLM cell requires an API key in `.env`.

Sections:
1. Setup & imports
2. Config & database
3. Add songs
4. Fetch lyrics
5. Phi metric (manual test)
6. Generate style description (LLM)
7. Generate phonetic variations (LLM)
8. Inspect accepted/rejected candidates
9. Export to PromptDataset JSON
10. Database status summary

## 1. Setup & imports

In [ ]:
import sys, logging, json
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(name)-20s %(levelname)-8s %(message)s')

# Change working directory to the repo root so relative paths work
import os
REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
os.chdir(REPO_ROOT)
print('Working directory:', REPO_ROOT)

from lyrkl.config import load_config
from lyrkl.db import Database
from lyrkl.pipeline import LyrkIPipeline
from lyrkl.phi import score_phi, filter_candidates
from lyrkl.models import VariantType, VariantStatus, StyleSource
from lyrkl.lyrics import clean_genius_lyrics, normalize_section_markers

## 2. Config & database

In [ ]:
config = load_config('configs/default.yaml')

# Use a separate test database so we don't pollute the main one
import dataclasses
config = dataclasses.replace(config, data_dir='data/test')

db = Database(config.db_path)
pipeline = LyrkIPipeline(config, db)

print('Config loaded:')
print(f'  LLM provider : {config.llm.provider}')
print(f'  LLM model    : {config.llm.model}')
print(f'  Phi threshold: {config.phi.min_aggregate}')
print(f'  DB path      : {config.db_path}')
print(f'  Genius key   : {"set" if config.genius_api_key() else "NOT SET"}')
print(f'  LLM key      : {"set" if config.llm_api_key() else "NOT SET"}')

## 3. Add songs

In [ ]:
TEST_SONGS = [
    {'title': 'Lose Yourself', 'artist': 'Eminem', 'genre': 'hip-hop'},
    {'title': 'Bohemian Rhapsody', 'artist': 'Queen', 'genre': 'rock'},
    {'title': 'Blinding Lights', 'artist': 'The Weeknd', 'genre': 'synth-pop'},
]

ids = pipeline.add_songs(TEST_SONGS)
print(f'Registered {len(ids)} songs:')
for sid in ids:
    print(f'  {sid}')

## 4. Fetch lyrics

Requires `GENIUS_API_KEY` to be set. Run the cell below to fetch for just the first song.

In [ ]:
# Fetch lyrics for Lose Yourself only
fetched = pipeline.fetch_lyrics(song_ids=['eminem__lose_yourself'])
print(f'Fetched: {fetched}')

song = db.get_song('eminem__lose_yourself')
if song and song.has_lyrics:
    lines = song.clean_lyrics.split('\n')
    # Print first 10 non-empty lines
    print(f'\nLyrics preview ({len(lines)} lines total):')
    count = 0
    for line in lines:
        if line.strip():
            print(f'  {line}')
            count += 1
            if count >= 10:
                print('  ...')
                break
else:
    print('No lyrics fetched. Check your GENIUS_API_KEY.')

## 5. Phi metric (manual test)

Test the 7-component Phi scoring on hand-crafted examples from the paper,
without needing an API key or Genius access.

In [ ]:
# Example from the paper's appendix:
# Original: "His palms are sweaty / Knees weak arms are heavy"
# APT:      "His palms are sweaty / Cheese weak cars are heavy"
# Expected: high Phi score

original = 'His palms are sweaty\nKnees weak arms are heavy'
apt      = 'His palms are sweaty\nCheese weak cars are heavy'
benign   = 'Walking through the park today\nThe sun is shining bright'

scores_apt    = score_phi(original, apt, config)
scores_benign = score_phi(original, benign, config)

def print_phi(label, s):
    print(f'\n{label}:')
    print(f'  phoneme       = {s.phoneme:.3f}  (S_ph)')
    print(f'  rhyme         = {s.rhyme:.3f}  (S_rh)')
    print(f'  syllable      = {s.syllable:.3f}  (S_sy)')
    print(f'  stress        = {s.stress:.3f}  (S_st)')
    print(f'  jaccard       = {s.jaccard:.3f}  (S_jac)')
    print(f'  cv_pattern    = {s.cv_pattern:.3f}  (S_cv)')
    print(f'  stressed_vowel= {s.stressed_vowel:.3f}  (S_vow)')
    print(f'  >>> AGGREGATE = {s.aggregate:.3f}  [threshold: {config.phi.min_aggregate}]')

print_phi('APT example (should be HIGH)', scores_apt)
print_phi('Benign example (should be LOW)', scores_benign)

In [ ]:
# Test filter_candidates
original_lines = 'His palms are sweaty\nKnees weak arms are heavy'
candidates = [
    'His palms are sweaty\nCheese weak cars are heavy',       # high phi
    'Walking through the park today\nThe sun is shining bright',  # low phi
]

accepted, rejected = filter_candidates(original_lines, candidates, config)
print(f'Accepted: {len(accepted)}')
for text, phi in accepted:
    print(f'  [{phi.aggregate:.3f}] {repr(text[:60])}')
print(f'Rejected: {len(rejected)}')
for text, phi in rejected:
    print(f'  [{phi.aggregate:.3f}] {repr(text[:60])}')

## 6. Generate style description (LLM)

Requires an LLM API key. Generates a one-sentence style descriptor using the model's world knowledge.

In [ ]:
done = pipeline.generate_style(song_ids=['eminem__lose_yourself'])
print(f'Generated style for: {done}')

style = db.get_latest_style('eminem__lose_yourself', StyleSource.LLM)
if style:
    print(f'\nSource: {style.source.value} / Model: {style.model}')
    print(f'Style: {style.text}')
else:
    print('No LLM style (check API key). Template fallback:')
    template_style = db.get_latest_style('eminem__lose_yourself', StyleSource.TEMPLATE)
    if template_style:
        print(f'  {template_style.text}')

## 7. Generate phonetic variations (LLM)

Calls the LLM with the APT prompt and filters candidates with Phi.
Requires a lyrics fetch (section 4) to have completed first.

In [ ]:
# Show the exact prompt that will be sent
from lyrkl.llm import build_apt_prompt

song = db.get_song('eminem__lose_yourself')
if song and song.has_lyrics:
    prompt_text, prompt_hash = build_apt_prompt(song, config)
    print(f'Prompt hash: {prompt_hash[:16]}...')
    print(f'Prompt length: {len(prompt_text)} chars')
    print('\n--- PROMPT PREVIEW (first 800 chars) ---')
    print(prompt_text[:800])
    print('...')
else:
    print('No lyrics found. Run the fetch cell first.')

In [ ]:
# Actually call the LLM and generate variations
summary = pipeline.generate_variations(song_ids=['eminem__lose_yourself'])
print('Variation summary:')
print(json.dumps(summary, indent=2))

## 8. Inspect accepted/rejected candidates

In [ ]:
song = db.get_song('eminem__lose_yourself')
accepted = db.get_variations('eminem__lose_yourself', status=VariantStatus.ACCEPTED)
rejected = db.get_variations('eminem__lose_yourself', status=VariantStatus.REJECTED)

print(f'Accepted: {len(accepted)}  |  Rejected: {len(rejected)}')

for i, var in enumerate(accepted):
    print(f'\n=== ACCEPTED #{i+1} (Phi_agg = {var.phi_aggregate:.3f}) ===')
    lines = var.lyrics.split('\n')
    orig_lines = (song.clean_lyrics if song else '').split('\n')
    for ol, ml in zip(orig_lines[:6], lines[:6]):
        ol_s = ol.strip()
        ml_s = ml.strip()
        if ol_s or ml_s:
            print(f'  ORIG: {ol_s}')
            print(f'  VAR:  {ml_s}')

if rejected:
    print(f'\n=== REJECTED #1 (Phi_agg = {rejected[0].phi_aggregate:.3f}) ===')
    print(rejected[0].lyrics[:300])

## 9. Export to PromptDataset JSON

In [ ]:
out_path = 'data/test/prompts.json'
n = pipeline.export_prompt_dataset(out_path)
print(f'Exported {n} records to {out_path}')

# Preview first record
with open(out_path) as f:
    records = json.load(f)

if records:
    r = records[0]
    print(f'\nSample record:')
    print(f'  song_id : {r["song_id"]}')
    print(f'  variant : {r["variant"]}')
    print(f'  caption : {r["caption"][:80] if r["caption"] else "(empty)"}')
    print(f'  phi_agg : {r["metadata"].get("phi_aggregate")}')
    print(f'  lyrics  : {r["lyrics"][:100]}...')

## 10. Database status summary

In [ ]:
summary = db.status_summary()
print(json.dumps(summary, indent=2))

In [ ]:
# List all LLM responses saved for this song
responses = db.list_llm_responses('eminem__lose_yourself')
print(f'LLM responses saved: {len(responses)}')
for resp in responses:
    print(f'  [{resp.purpose}] model={resp.model} candidates={resp.n_candidates}')
    print(f'    hash={resp.prompt_hash[:16]}...')
    print(f'    file={resp.raw_file_path}')